# Chapter 2 — Computing with vectors and matrices · Notebook to complete

*Maths for AI, step by step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch02_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells to edit are marked **TO COMPLETE**. Run the cells in order: as long as an answer is missing, the checks say so without an error. The solution is in `ch02_corrige.ipynb`, on the same companion page.

**Prediction rule of the chapter** (given, not fitted): weights $\mathbf{w} = (3 ; -8)$ in k€/m² and k€/km, intercept $b = 45$ k€.

With the neighbourhoods (lesson 2.4), the rule becomes $\mathbf{w} = (3 ; -8 ; 12 ; -10)$, $b = 45$ k€, with Gare (the station) as the reference neighbourhood.

**Contents.** Data · The lesson computations · Exercises 2.1 to 2.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colours of the book: one colour = one role (observed, predicted, deviation, colleague's line)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifiers (D2.1, D2.E2) are those of the book and of the CSV files.

Variable and column names in the code are in French: `logement` = home, `surface_m2` = floor area in m², `distance_km` = distance to the centre in km, `prix_kEUR` = price in k€, `quartier` = neighbourhood (Parc = park, Rocade = ring road, Gare = station), `est` = east, `t2_sud` = two-room flats in the South, `vente` = sale, `nouveaux` = new homes, `poids` = weights, `ecart` / `eam` = deviation / mean absolute deviation.

In [ ]:
# Data of chapter 2, copied from the book's CSV files (do not modify)

# D2.1 — the seven homes with their distance to the centre (lessons 2.1 to 2.3)
logements = pd.DataFrame({
    'logement':    ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'surface_m2':  [32, 45, 54, 70, 85, 98, 62],
    'distance_km': [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'prix_kEUR':   [118, 172, 185, 245, 279, 335, 209],
})

# D2.4 — five homes in the east of the town, with their neighbourhood (lesson 2.4)
est = pd.DataFrame({
    'logement':    ['K', 'L', 'M', 'N', 'O'],
    'quartier':    ['Parc', 'Rocade', 'Gare', 'Parc', 'Rocade'],
    'surface_m2':  [60, 60, 60, 45, 80],
    'distance_km': [2.0, 2.0, 2.0, 1.0, 3.0],
    'prix_kEUR':   [223, 200, 210, 186, 248],
})

# D2.5 — the five two-room flats (T2) in the South with their distance to the centre (lesson 2.5)
t2_sud = pd.DataFrame({
    'vente':       ['S03', 'S04', 'S05', 'S06', 'S07'],
    'surface_m2':  [43, 40, 44, 46, 52],
    'distance_km': [3.5, 2.0, 1.5, 2.5, 0.5],
})

# D2.E2 — three homes put up for sale (exercise 2.2)
nouveaux = pd.DataFrame({
    'logement':    ['H', 'I', 'J'],
    'surface_m2':  [40, 75, 90],
    'distance_km': [0.5, 3.5, 1.0],
})

## The lesson computations, in Python

These cells redo the computations of lessons 2.1 to 2.5. Run them and compare with the book.

In [ ]:
# Lesson 2.1 — the agency's rule applied to G, term by term (equation 2.1)
part_surface = 3 * 62        # k€/m² × m² = k€
part_distance = -8 * 2.5     # k€/km × km = k€
part_fixe = 45               # k€
prix_G = part_surface + part_distance + part_fixe
print(part_surface, part_distance, part_fixe, '→', prix_G, 'k€; signed deviation:', 209 - prix_G, 'k€')

In [ ]:
# Lesson 2.2 — vectors, dot product (equation 2.2) and prediction (equation 2.3)
x_G = np.array([62, 2.5])    # (floor area in m², distance in km)
w = np.array([3, -8])        # (k€/m², k€/km), same order as x
b = 45                       # k€
print('dimension of x_G:', x_G.shape, '  w · x_G =', np.dot(w, x_G), '  ŷ =', np.dot(w, x_G) + b)
print('weights in the wrong order:', np.dot(w[::-1], x_G) + b, 'k€ (absurd)')

In [ ]:
# Lesson 2.3 — all predictions at once (equation 2.5)
X = logements[['surface_m2', 'distance_km']].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
y_chapeau = X @ w + b
print('X :', X.shape, ' w :', w.shape, ' ŷ :', y_chapeau.shape)
print('ŷ =', y_chapeau)
print('signed deviations y − ŷ =', y - y_chapeau)
print('mean absolute deviation, 7 homes:', round(np.mean(np.abs(y - y_chapeau)), 2), 'k€')
print('same, A to F:', round(np.mean(np.abs(y - y_chapeau)[:6]), 2), 'k€; line from the chapter opening, A to F:',
      round(np.mean(np.abs(y - (3 * X[:, 0] + 30))[:6]), 2), 'k€')

In [ ]:
# Lesson 2.4 — indicator columns (equation 2.6), Gare as the reference
indicatrices = pd.get_dummies(est['quartier'], drop_first=True).astype(int)   # columns Parc, Rocade
X_est = np.column_stack([est['surface_m2'], est['distance_km'], indicatrices['Parc'], indicatrices['Rocade']])
w_est = np.array([3, -8, 12, -10])
y_est = est['prix_kEUR'].to_numpy(dtype=float)
print(indicatrices.columns.tolist(), ' X :', X_est.shape)
print('with the neighbourhood:', X_est @ w_est + 45)
print('without the neighbourhood:', X_est[:, :2] @ w_est[:2] + 45)
print('mean absolute deviation: without', np.mean(np.abs(y_est - (X_est[:, :2] @ w_est[:2] + 45))),
      ' with', np.mean(np.abs(y_est - (X_est @ w_est + 45))))

In [ ]:
# Lesson 2.5 — the same prediction on two scales (equation 2.7), division by n
X5 = t2_sud[['surface_m2', 'distance_km']].to_numpy(dtype=float)
mu, sigma = X5.mean(axis=0), X5.std(axis=0, ddof=0)
w, b = np.array([3, -8]), 45
w_std, b_std = w * sigma, b + np.sum(w * mu)
Z = (X5 - mu) / sigma
print('means:', mu, ' standard deviations:', sigma)
print('standardised weights:', w_std, ' intercept:', b_std)
print('original units:', X5 @ w + b)
print('standardised scale:', Z @ w_std + b_std)

## Exercise 2.1 — Reading a diagram

*Concepts: product of a matrix and a vector (lesson 2.3).* Go back to the matrix diagram of lesson 2.3 (the seven homes A to G). The cell below rebuilds $X$, $\mathbf{w}$ and $b$ to check your answers.

In [ ]:
X = logements[['surface_m2', 'distance_km']].to_numpy(dtype=float)
w, b = np.array([3, -8]), 45
print(X)

In [ ]:
# TO COMPLETE: your answers, read from the diagram
dimensions = {'X': None, 'w': None, 'y_chapeau': None}   # e.g. (7, 2), (2,), (7,)
ligne_280 = None             # letter of the home whose prediction is 280 k€
dimensions_c = {'X': None, 'w': None, 'y_chapeau': None}   # with the floor level and an 8th home

In [ ]:
# Check
y_chapeau = X @ w + b
vrai = {'X': X.shape, 'w': w.shape, 'y_chapeau': y_chapeau.shape}
if None in dimensions.values() or ligne_280 is None or None in dimensions_c.values():
    print('Complete the previous cell.')
else:
    print('a) dimensions:', 'correct' if dimensions == vrai else 'check again')
    lettre = logements['logement'][int(np.argmin(np.abs(y_chapeau - 280)))]
    print('b) home:', 'correct' if ligne_280 == lettre else 'check again')
    X8 = np.zeros((8, 3)); w3 = np.zeros(3)   # shape only: 8 homes, 3 features
    print('c) dimensions:', 'correct' if dimensions_c == {'X': X8.shape, 'w': w3.shape,
          'y_chapeau': (X8 @ w3).shape} else 'check again')

*Your explanations:* …

## Exercise 2.2 — Three new homes

*Data D2.E2. Concepts: dot product (lesson 2.2), product of a matrix and a vector (lesson 2.3).* Do questions a) to c) on paper first, with the table of exercise 2.2 in the book.

In [ ]:
nouveaux

### a) and b) On paper

Write the three vectors, the matrix $X$ and its dimensions, then the three predicted prices with their units. Enter your prices below.

In [ ]:
# TO COMPLETE: your predicted prices computed by hand, in k€
prix_main = {'H': None, 'I': None, 'J': None}

In [ ]:
# Check (independent computation, row by row)
for _, l in nouveaux.iterrows():
    vrai = 3 * l['surface_m2'] - 8 * l['distance_km'] + 45
    la_votre = prix_main[l['logement']]
    etat = 'not yet completed' if la_votre is None else ('correct' if np.isclose(la_votre, vrai) else 'check again')
    print(f"{l['logement']} : {etat}")

### c) Interpret

*Your answer:* …

### d) All predictions in one line

Complete the cell, then run the check.

In [ ]:
# X: one row per home (H, I, J),
#    columns floor area then distance
X = None          # TO COMPLETE
w = None          # TO COMPLETE: order of the columns of X
b = None          # TO COMPLETE: in k€
y_chapeau = None  # TO COMPLETE: equation 2.5

<details><summary><b>Hint</b> (click if you are stuck)</summary>

`nouveaux[['surface_m2', 'distance_km']].to_numpy()` gives the matrix. `np.array([..., ...])` creates a vector. The matrix product is written `@`.

</details>

In [ ]:
# Check
if y_chapeau is None:
    print('Complete the previous cell.')
else:
    print('dimensions:', np.shape(X), '@', np.shape(w), '→', np.shape(y_chapeau))
    print('dimensions:', 'correct' if np.shape(X) == (3, 2) and np.shape(y_chapeau) == (3,) else 'check again')
    print('predictions:', y_chapeau, '→',
          'correct' if np.allclose(y_chapeau, [161, 242, 307]) else 'check again')

### e) Predict, then check

Columns of $X$ in the order distance then floor area, $\mathbf{w}$ unchanged. First write down your prediction of the sign of the three prices, then run the cell.

*Your prediction:* …

In [ ]:
if y_chapeau is not None:
    X_inverse = X[:, ::-1]          # columns swapped: distance, floor area
    print('columns swapped:', X_inverse @ w + b)

## Exercise 2.3 — What do the weights say?

*Concepts: reading a weight (lesson 2.2), indicator columns (lesson 2.4).* Rule: $\mathbf{w} = (3 ; -8 ; 12 ; -10)$ for floor area, distance, Parc, Rocade; $b = 45$ k€; Gare is the reference. Questions on paper, check below.

In [ ]:
# TO COMPLETE: differences in predicted price, in k€ (first home − second)
a_parc_moins_rocade = None
b_plus_loin_moins_proche = None   # 1.5 km farther, same neighbourhood, same floor area
c_parc_loin_moins_gare = None     # Parc and 1.5 km farther, compared with Gare

In [ ]:
# Check: we compute the rule for pairs of fictitious homes
regle = lambda surface, distance, parc, rocade: 3 * surface - 8 * distance + 12 * parc - 10 * rocade + 45
vrais = {'a': regle(60, 2, 1, 0) - regle(60, 2, 0, 1),
         'b': regle(60, 3.5, 0, 0) - regle(60, 2, 0, 0),
         'c': regle(60, 3.5, 1, 0) - regle(60, 2, 0, 0)}
for k, la_votre in [('a', a_parc_moins_rocade), ('b', b_plus_loin_moins_proche), ('c', c_parc_loin_moins_gare)]:
    etat = 'not yet completed' if la_votre is None else ('correct' if np.isclose(la_votre, vrais[k]) else 'check again')
    print(f'{k}) {etat}')

*Your interpretations for c) and d):* …

## Exercise 2.4 — Indicator columns in Python

*Data D2.4. Concepts: indicator columns (lesson 2.4), product of a matrix and a vector (lesson 2.3).*

In [ ]:
est

In [ ]:
# TO COMPLETE
parc = None     # a) 1 if the home is in Parc, 0 otherwise (integers)
rocade = None   # a) 1 if the home is in Rocade, 0 otherwise
X4 = None       # b) 5 × 4: floor area, distance, parc, rocade
w4 = np.array([3, -8, 12, -10])
b4 = 45
y4 = None       # b) equation 2.5

<details><summary><b>Hint</b> (click if you are stuck)</summary>

`(est['quartier'] == 'Parc')` gives true/false values; `.astype(int)` turns them into 1/0. `np.column_stack([a, b, c, d])` puts four columns side by side.

</details>

In [ ]:
# Check
if y4 is None:
    print('Complete the previous cell.')
else:
    print('indicator columns:', 'correct' if list(parc) == [1, 0, 0, 1, 0] and list(rocade) == [0, 1, 0, 0, 1] else 'check again')
    print('X4 :', np.shape(X4), '→', 'correct' if np.shape(X4) == (5, 4) else 'check again')
    print('predictions:', y4, '→', 'correct' if np.allclose(y4, [221, 199, 209, 184, 251]) else 'check again')

**c)** Mean absolute deviation, with and without the neighbourhood columns.

In [ ]:
# TO COMPLETE
y_obs = est['prix_kEUR'].to_numpy(dtype=float)
eam_sans = None   # without neighbourhood: first two columns and first two weights
eam_avec = None

In [ ]:
if eam_sans is None or eam_avec is None:
    print('Complete the previous cell.')
else:
    print(f'without: {eam_sans:.1f} k€, with: {eam_avec:.1f} k€ →',
          'correct' if np.isclose(eam_sans, 10.2) and np.isclose(eam_avec, 1.8) else 'check again')

*Your comment:* …

## Exercise 2.5 — Predict, then check: changing units

*Data D2.5. Concepts: change of scale (lesson 2.5, equation 2.7).* Write down your predictions for a) and b) before running the cells.

*Your predictions:* …

In [ ]:
# TO COMPLETE: conversion to standardised weights (equation 2.7)
def poids_standardises(w, b, mu, sigma):
    w_std = None   # TO COMPLETE
    b_std = None   # TO COMPLETE
    return w_std, b_std

In [ ]:
# Two versions of the same rule: distance in km, then in metres
X_km = t2_sud[['surface_m2', 'distance_km']].to_numpy(dtype=float)
X_m = X_km * np.array([1, 1000])
w_km, w_m, b = np.array([3, -8]), np.array([3, -0.008]), 45
print('predictions (km):', X_km @ w_km + b)
print('predictions (m): ', X_m @ w_m + b)
resultats = {}
for nom, Xv, wv in [('km', X_km, w_km), ('m', X_m, w_m)]:
    mu, sigma = Xv.mean(axis=0), Xv.std(axis=0, ddof=0)
    resultats[nom] = poids_standardises(wv, b, mu, sigma)
    print(nom, ': mean distance =', mu[1], ', standard deviation =', sigma[1], ', standardised weights =', resultats[nom])

In [ ]:
# Check
if resultats['km'][0] is None:
    print('Complete the function poids_standardises.')
else:
    ok = all(np.allclose(resultats[k][0], [12, -8]) and np.isclose(resultats[k][1], 164) for k in resultats)
    print('standardised weights identical in km and in m:', 'correct' if ok else 'check again')

In [ ]:
if resultats['km'][0] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
    for ax, titre, valeurs in [
            (axes[0], 'Raw weight of the 2nd term', {'km': w_km[1], 'm': w_m[1]}),
            (axes[1], 'Standardised weight of the 2nd term', {k: resultats[k][0][1] for k in resultats})]:
        ax.bar(list(valeurs), list(valeurs.values()), color=[BLEU, ORANGE])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(titre, fontsize=10)
    plt.tight_layout()
    plt.show()

*Your conclusion for c):* …

## Exercise 2.6 — Challenge: one rule, several ways to write it

*Data D2.4. Concepts: lessons 2.3, 2.4 and 2.5.* Three indicator columns: Parc, Rocade, Gare.

In [ ]:
gare = (est['quartier'] == 'Gare').astype(int).to_numpy()
parc3 = (est['quartier'] == 'Parc').astype(int).to_numpy()
rocade3 = (est['quartier'] == 'Rocade').astype(int).to_numpy()
X3 = np.column_stack([est['surface_m2'], est['distance_km'], parc3, rocade3, gare])
print('sum of the three indicators, row by row:', X3[:, 2:].sum(axis=1))
print('colleague\'s rule (b = 40):', X3 @ np.array([3, -8, 17, -5, 5]) + 40)

In [ ]:
# TO COMPLETE b): three neighbourhood weights (Parc, Rocade, Gare) for b = 50
poids_quartier_b50 = None   # list of three numbers

In [ ]:
# Check
if poids_quartier_b50 is None:
    print('Complete the previous cell.')
else:
    y50 = X3 @ np.array([3, -8] + list(poids_quartier_b50)) + 50
    print('predictions with b = 50:', y50, '→', 'correct' if np.allclose(y50, [221, 199, 209, 184, 251]) else 'check again')

*Your answer to c):* …